# AI Trader — LSTM Autoencoder for Market Anomaly Detection

**Purpose:** Train a sequence-to-sequence LSTM autoencoder on NSE stock OHLCV data.  
High reconstruction error = unusual market behaviour = anomaly suppresses signals.

**Colab setup:**
1. `Runtime → Change runtime type → T4 GPU` (free tier is fine)
2. `Runtime → Mount Drive` OR the code will mount it automatically
3. Run all cells (Ctrl+F9)
4. Copy the **Google Drive File ID** printed in the last cell
5. Add it to your `.env`: `LSTM_GDRIVE_ID=<paste_here>`

**Training time:** ~5–10 min on T4 GPU for 50 symbols × 500 days.

In [ ]:
# ── Install / upgrade dependencies ─────────────────────────────────────────
!pip install -q yfinance torch numpy pandas tqdm

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  — edit these values before running
# ══════════════════════════════════════════════════════════════════════════════

# Google Drive folder where the artifact will be saved
DRIVE_SAVE_DIR = '/content/drive/MyDrive/ai_trader_models'

# Training window: how many calendar days of history to pull per symbol
HISTORY_PERIOD = '3y'

# Model hyperparameters (must stay in sync with backend/app/services/lstm_service.py)
SEQ_LEN     = 30    # rolling window length in trading days
INPUT_SIZE  = 4     # features per timestep
HIDDEN_SIZE = 64
NUM_LAYERS  = 2

# Training
EPOCHS      = 60
BATCH_SIZE  = 64
LR          = 1e-3

# Anomaly threshold percentile (higher = less sensitive)
THRESHOLD_PERCENTILE = 95

# Minimum bars required after feature engineering to include a symbol
MIN_BARS = SEQ_LEN + 25

# Cap total symbols (None = all qualifying; reduce if you hit Colab timeout)
MAX_SYMBOLS = None


In [ ]:
# ── Fetch full NSE equity universe from NSE public CSV ───────────────────────
# NSE lists ~1800+ actively traded equities. We download all of them and filter
# to those with at least MIN_BARS of price history (set in CONFIG above).
# Typical qualifying count after filtering: 600–900 symbols.
# Colab T4 GPU training time: ~20–40 min (LSTM), ~35–60 min (TFT).
import io as _io, urllib.request as _req
import pandas as pd

NSE_EQUITY_CSV = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"

try:
    _r = _req.Request(NSE_EQUITY_CSV, headers={"User-Agent": "Mozilla/5.0"})
    with _req.urlopen(_r, timeout=30) as resp:
        content = resp.read().decode("utf-8", errors="ignore")
    df_universe = pd.read_csv(_io.StringIO(content))
    raw_syms = df_universe["SYMBOL"].dropna().str.strip().tolist()
    SYMBOLS  = [s + ".NS" for s in raw_syms if s]
    print(f"NSE universe: {len(SYMBOLS)} symbols fetched from NSE CSV")
except Exception as _e:
    print(f"WARNING: NSE CSV unavailable ({_e}). Using ~200-symbol fallback list.")
    _FALLBACK = [
        "RELIANCE","TCS","HDFCBANK","INFY","ICICIBANK","HINDUNILVR","ITC","SBIN",
        "BHARTIARTL","KOTAKBANK","LT","AXISBANK","ASIANPAINT","MARUTI","SUNPHARMA",
        "WIPRO","ULTRACEMCO","TITAN","BAJFINANCE","ONGC","NTPC","POWERGRID",
        "NESTLEIND","TECHM","HCLTECH","INDUSINDBK","COALINDIA","GRASIM","JSWSTEEL",
        "TATAMOTORS","ADANIENT","ADANIPORTS","BAJAJ-AUTO","BPCL","CIPLA","DIVISLAB",
        "DRREDDY","EICHERMOT","HEROMOTOCO","HINDALCO","M&M","BRITANNIA","APOLLOHOSP",
        "TATASTEEL","UPL","VEDL","BAJAJFINSV","HDFCLIFE","SBILIFE","PIDILITIND",
        "HAVELLS","MUTHOOTFIN","IDFCFIRSTB","BANKBARODA","PNB","CANBK","FEDERALBNK",
        "BANDHANBNK","RBLBANK","SIEMENS","ABB","VOLTAS","AMBUJACEM","SHREECEM",
        "DALMIACEM","JKCEMENT","ZOMATO","NYKAA","IRCTC","DELHIVERY","NAUKRI",
        "TRENT","ABFRL","PAGEIND","MANYAVAR","BATAINDIA","METRO","RELAXO",
        "GLAND","GRANULES","NATCO","LALPATHLAB","METROPOLIS","THYROCARE","ALKEM",
        "AUROPHARMA","TORNTPHARM","IPCALAB","PFIZER","GLAXO","ABBOTINDIA","BIOCON",
        "TATACHEM","DEEPAKNTR","NAVINFLUOR","NOCIL","SRF","ATUL","GHCL",
        "AAVAS","HOMEFIRST","APTUS","CANFINHOME","LICHSGFIN","PNBHOUSING",
        "MAHINDCIE","MOTHERSON","BOSCHLTD","BHARATFORG","SUNDARMFIN","CUMMINSIND",
        "THERMAX","ESCORTS","SONACOMS","MINDA","ENDURANCE","GABRIEL",
        "MPHASIS","LTIM","PERSISTENT","COFORGE","CYIENT","KPIT","OFSS",
        "INTERGLOBE","CONCOR","BLUEDART","TATACOMM","ZEEL","SUNTV","STAR",
        "LICI","PEL","CHOLAFIN","MANAPPURAM","M&MFIN","SRTRANSFIN",
        "SHRIRAMFIN","ICICIGI","ICICIPRU","HDFCAMC","NIPPONLIFE",
        "ANGELONE","MOTILALOFS","EDELWEISS","LAURUSLABS","DIVIS","GLENMARK",
        "APOLLOHOSP","FORTIS","MAXHEALTH","NARAYANA","RAINBOW",
    ]
    SYMBOLS = [s + ".NS" for s in _FALLBACK]

if MAX_SYMBOLS:
    SYMBOLS = SYMBOLS[:MAX_SYMBOLS]

print(f"Will attempt to download: {len(SYMBOLS)} symbols")
print("Symbols with < MIN_BARS of data will be skipped automatically.")


In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os, math, time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import yfinance as yf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# ── Fetch OHLCV data ─────────────────────────────────────────────────────────
print(f'Fetching {len(SYMBOLS)} symbols...')
all_data = {}
failed   = []

for sym in tqdm(SYMBOLS):
    try:
        df = yf.download(sym, period=HISTORY_PERIOD,
                         interval='1d', auto_adjust=True, progress=False)
        df.dropna(inplace=True)
        if len(df) >= SEQ_LEN + 25:
            all_data[sym] = df
        else:
            failed.append(sym)
    except Exception as e:
        failed.append(sym)
        print(f'  WARN: {sym} — {e}')

print(f'\nLoaded {len(all_data)} symbols  |  Failed/thin: {failed}')

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
# Features (must match backend/app/services/lstm_service.py _build_sequence):
#   0  close_pct      — daily return
#   1  hl_ratio       — (high − low) / close
#   2  vol_ratio      — volume / 20-day rolling mean volume
#   3  close_vs_ma20  — (close − SMA20) / SMA20

def build_features(df: pd.DataFrame) -> np.ndarray:
    closes  = df['Close'].values.squeeze().astype(np.float32)
    highs   = df['High'].values.squeeze().astype(np.float32)
    lows    = df['Low'].values.squeeze().astype(np.float32)
    volumes = df['Volume'].values.squeeze().astype(np.float32) + 1e-8

    close_pct     = np.diff(closes) / (closes[:-1] + 1e-8)
    hl_ratio      = (highs[1:] - lows[1:]) / (closes[1:] + 1e-8)
    vol_ma20      = pd.Series(volumes).rolling(20, min_periods=1).mean().values[1:]
    vol_ratio     = volumes[1:] / (vol_ma20 + 1e-8)
    ma20          = pd.Series(closes).rolling(20, min_periods=1).mean().values[1:]
    close_vs_ma20 = (closes[1:] - ma20) / (ma20 + 1e-8)

    return np.stack([close_pct, hl_ratio, vol_ratio, close_vs_ma20], axis=1)  # (T, 4)


def make_sequences(features: np.ndarray, seq_len: int) -> np.ndarray:
    seqs = []
    for i in range(len(features) - seq_len + 1):
        seqs.append(features[i : i + seq_len])
    return np.stack(seqs, axis=0)   # (N, seq_len, 4)


all_seqs = []
for sym, df in all_data.items():
    feat = build_features(df)
    seqs = make_sequences(feat, SEQ_LEN)
    all_seqs.append(seqs)

all_seqs = np.concatenate(all_seqs, axis=0).astype(np.float32)
print(f'Total sequences: {len(all_seqs):,}  shape: {all_seqs.shape}')

In [ ]:
# ── Standardise ─────────────────────────────────────────────────────────────
flat         = all_seqs.reshape(-1, INPUT_SIZE)
scaler_mean  = flat.mean(axis=0)
scaler_std   = flat.std(axis=0) + 1e-8
all_seqs_norm = (all_seqs - scaler_mean) / scaler_std

# Train / val split (80 / 20)
n_train = int(len(all_seqs_norm) * 0.8)
train_X = torch.tensor(all_seqs_norm[:n_train])
val_X   = torch.tensor(all_seqs_norm[n_train:])

train_loader = DataLoader(TensorDataset(train_X), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_X),   batch_size=BATCH_SIZE)

print(f'Train: {len(train_X):,}  Val: {len(val_X):,}')

In [ ]:
# ── Model definition ─────────────────────────────────────────────────────────
# MUST be identical to backend/app/services/lstm_service.py _build_model
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        drop = 0.1 if num_layers > 1 else 0.0
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers,
                               batch_first=True, dropout=drop)
        self.decoder = nn.LSTM(hidden_size, hidden_size, num_layers,
                               batch_first=True, dropout=drop)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        dec_in    = h[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        dec_out, _ = self.decoder(dec_in)
        return self.output_layer(dec_out)

model = LSTMAutoencoder(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, SEQ_LEN).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                 optimizer, patience=5, factor=0.5, verbose=True)
criterion  = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in tqdm(range(1, EPOCHS + 1)):
    # --- train ---
    model.train()
    t_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), xb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item() * len(xb)
    t_loss /= len(train_X)

    # --- val ---
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(DEVICE)
            v_loss += criterion(model(xb), xb).item() * len(xb)
    v_loss /= len(val_X)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), '/tmp/lstm_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}  train={t_loss:.6f}  val={v_loss:.6f}')

# Reload best weights
model.load_state_dict(torch.load('/tmp/lstm_best.pt', map_location=DEVICE))
print(f'\nBest val loss: {best_val_loss:.6f}')

In [ ]:
# ── Plot loss curves ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE')
plt.plot(val_losses,   label='Val MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend()
plt.title('LSTM Autoencoder — Training Loss')
plt.tight_layout(); plt.show()

In [ ]:
# ── Compute anomaly threshold ─────────────────────────────────────────────────
# Threshold = Nth-percentile reconstruction MSE over the TRAINING set
model.eval()
mse_list = []

with torch.no_grad():
    for (xb,) in DataLoader(TensorDataset(train_X), batch_size=512):
        xb   = xb.to(DEVICE)
        diff = (model(xb) - xb) ** 2
        # per-sample mean MSE
        mse_list.extend(diff.mean(dim=(1, 2)).cpu().numpy().tolist())

mse_arr   = np.array(mse_list)
threshold = float(np.percentile(mse_arr, THRESHOLD_PERCENTILE))

print(f'Threshold ({THRESHOLD_PERCENTILE}th pct) = {threshold:.8f}')
print(f'Max MSE on train: {mse_arr.max():.8f}')

plt.figure(figsize=(9, 3))
plt.hist(mse_arr, bins=100, log=True)
plt.axvline(threshold, color='red', label=f'{THRESHOLD_PERCENTILE}th pct')
plt.xlabel('MSE'); plt.ylabel('Count (log)'); plt.legend()
plt.title('Reconstruction MSE Distribution')
plt.tight_layout(); plt.show()

In [ ]:
# ── Save artifact to Google Drive ─────────────────────────────────────────────
import os, re

VERSION   = f"lstm-ae-v{datetime.now().strftime('%Y%m%d-%H%M')}"
SAVE_PATH = f'{DRIVE_SAVE_DIR}/lstm_autoencoder_latest.pt'

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

model.to('cpu')
payload = {
    'version':       VERSION,
    'config': {
        'input_size':  INPUT_SIZE,
        'hidden_size': HIDDEN_SIZE,
        'num_layers':  NUM_LAYERS,
        'seq_len':     SEQ_LEN,
    },
    'state_dict':    model.state_dict(),
    'scaler_mean':   scaler_mean.tolist(),
    'scaler_std':    scaler_std.tolist(),
    'threshold':     threshold,
    'feature_names': ['close_pct', 'hl_ratio', 'vol_ratio', 'close_vs_ma20'],
    'metrics': {
        'best_val_mse':  round(best_val_loss, 8),
        'threshold_pct': THRESHOLD_PERCENTILE,
        'n_symbols':     len(all_data),
        'n_sequences':   len(all_seqs),
    },
}
torch.save(payload, SAVE_PATH)
print(f'Saved to: {SAVE_PATH}')
print(f'File size: {os.path.getsize(SAVE_PATH) / 1024:.1f} KB')

In [ ]:
# ── Get Google Drive file ID ──────────────────────────────────────────────────
# Share the file so the backend container can download it with gdown.

from google.colab import auth
from googleapiclient.discovery import build
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
creds   = GoogleCredentials.get_application_default()
service = build('drive', 'v3', credentials=creds)

# Search for the file by name
results = service.files().list(
    q=f"name='lstm_autoencoder_latest.pt'",
    fields='files(id, name, webViewLink)',
).execute()

files = results.get('files', [])
if files:
    file_id   = files[0]['id']
    view_link = files[0]['webViewLink']

    # Make it publicly readable so gdown can download without OAuth
    service.permissions().create(
        fileId=file_id,
        body={'type': 'anyone', 'role': 'reader'},
    ).execute()

    print('=' * 60)
    print('LSTM Autoencoder artifact saved and shared.')
    print(f'\nGoogle Drive File ID: {file_id}')
    print(f'View link:           {view_link}')
    print('\n>>> Add to your .env file:')
    print(f'LSTM_GDRIVE_ID={file_id}')
    print('\nThen run inside the backend container:')
    print('  docker compose exec backend python scripts/download_models.py --lstm-only')
    print('  docker compose restart backend celery-worker')
    print('=' * 60)
else:
    print('File not found in Drive — check DRIVE_SAVE_DIR path above.')